In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from global_vars import *
import gc
import os
from sklearn.model_selection import RandomizedSearchCV
import fastparquet
import joblib



train_folder = os.path.join(data_folder, 'nodeep_train_data_ver1/')
test_folder = os.path.join(data_folder, 'nodeep_test_data_ver1/')


X_train = pd.read_parquet(os.path.join(train_folder, "X_train.parquet"),engine="fastparquet")
y_t_train = pd.read_parquet(os.path.join(train_folder, "y_t_train.parquet"),engine="fastparquet")
y_q_train = pd.read_parquet(os.path.join(train_folder, "y_q_train.parquet"),engine="fastparquet")

X_test = pd.read_parquet(os.path.join(test_folder, "X_test.parquet"),engine="fastparquet")
y_t_test = pd.read_parquet(os.path.join(test_folder, "y_t_test.parquet"),engine="fastparquet")
y_q_test = pd.read_parquet(os.path.join(test_folder, "y_q_test.parquet"),engine="fastparquet")

In [6]:
for mf in ["sqrt", 0.2, 0.3, 0.5, 1.0]:
    rf = RandomForestRegressor(n_estimators=50, max_features=mf,
                               max_depth=20, n_jobs=-1,
                               oob_score=True, random_state=0)
    rf.fit(X_train, y_q_train)
    print(f"max_features={mf}: OOB R²={rf.oob_score_:.4f}")

max_features=sqrt: OOB R²=0.3804
max_features=0.2: OOB R²=0.4005
max_features=0.3: OOB R²=0.4025
max_features=0.5: OOB R²=0.4028
max_features=1.0: OOB R²=0.4013


In [11]:
rf = RandomForestRegressor(n_estimators=250, random_state=0, max_depth=30, verbose=2, n_jobs=-1, max_features=0.5)
rf.fit(X_train, y_q_train)

## 150, 20, 0.5 --> 0.352
## 150, 30, 0.5 --> 0.356
## 140, 30, 0.5 --> 0.34
# 100, 30 --> R^2: 0.35

pred = rf.predict(X_test)           # -> (n_test, 3)
print(pred.shape)
try:
    print("R^2:", rf.score(X_test, y_q_test))
except:
    print('woops')


np.save(os.path.join(test_folder, "y_q_pred_rf200_50.npy"), pred)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 25 concurrent workers.


building tree 1 of 250
building tree 2 of 250
building tree 3 of 250
building tree 4 of 250
building tree 5 of 250
building tree 6 of 250
building tree 7 of 250
building tree 8 of 250
building tree 9 of 250
building tree 10 of 250
building tree 11 of 250
building tree 12 of 250
building tree 13 of 250
building tree 14 of 250
building tree 15 of 250
building tree 16 of 250
building tree 17 of 250
building tree 18 of 250
building tree 19 of 250
building tree 20 of 250
building tree 21 of 250
building tree 22 of 250
building tree 23 of 250
building tree 24 of 250
building tree 25 of 250
building tree 26 of 250
building tree 27 of 250
building tree 28 of 250
building tree 29 of 250
building tree 30 of 250
building tree 31 of 250
building tree 32 of 250
building tree 33 of 250
building tree 34 of 250
building tree 35 of 250
building tree 36 of 250
building tree 37 of 250
building tree 38 of 250
building tree 39 of 250
building tree 40 of 250
building tree 41 of 250
building tree 42 of 250
b

[Parallel(n_jobs=-1)]: Done 112 tasks      | elapsed: 32.6min


building tree 138 of 250
building tree 139 of 250
building tree 140 of 250
building tree 141 of 250
building tree 142 of 250
building tree 143 of 250
building tree 144 of 250
building tree 145 of 250
building tree 146 of 250
building tree 147 of 250
building tree 148 of 250
building tree 149 of 250
building tree 150 of 250
building tree 151 of 250
building tree 152 of 250
building tree 153 of 250
building tree 154 of 250
building tree 155 of 250
building tree 156 of 250
building tree 157 of 250
building tree 158 of 250
building tree 159 of 250
building tree 160 of 250
building tree 161 of 250
building tree 162 of 250
building tree 163 of 250
building tree 164 of 250
building tree 165 of 250
building tree 166 of 250
building tree 167 of 250
building tree 168 of 250
building tree 169 of 250
building tree 170 of 250
building tree 171 of 250
building tree 172 of 250
building tree 173 of 250
building tree 174 of 250
building tree 175 of 250
building tree 176 of 250
building tree 177 of 250


[Parallel(n_jobs=-1)]: Done 250 out of 250 | elapsed: 69.5min finished
[Parallel(n_jobs=25)]: Using backend ThreadingBackend with 25 concurrent workers.
[Parallel(n_jobs=25)]: Done 112 tasks      | elapsed:    7.7s
[Parallel(n_jobs=25)]: Done 250 out of 250 | elapsed:   15.7s finished


(2138004, 58)


[Parallel(n_jobs=25)]: Using backend ThreadingBackend with 25 concurrent workers.
[Parallel(n_jobs=25)]: Done 112 tasks      | elapsed:    7.8s
[Parallel(n_jobs=25)]: Done 250 out of 250 | elapsed:   15.9s finished


R^2: 0.3577822744273224


In [14]:
for i in (np.array(range(4, 16)) * 0.05):
    print(i)
    rf = RandomForestRegressor(n_estimators=150, random_state=0, max_depth=30, n_jobs=-1, max_features=i)
    rf.fit(X_train, y_q_train)
    
    ## 150, 20, 0.5 --> 0.352
    
    pred = rf.predict(X_test)           # -> (n_test, 3)
    print(pred.shape)
    try:
        print("R^2:", rf.score(X_test, y_q_test))
    except:
        print('woops')
    
    
    np.save(os.path.join(test_folder, "y_q_pred_rf200_50.npy"), pred)

0.2
(2138004, 58)
R^2: 0.3497336374848494
0.25
(2138004, 58)
R^2: 0.3507010112343527
0.30000000000000004


KeyboardInterrupt: 

0.2
(2138004, 58)
R^2: 0.3456217385469707
0.25
(2138004, 58)
R^2: 0.33575315747052803
0.30000000000000004
(2138004, 58)
R^2: 0.35148356105040685
0.35000000000000003
(2138004, 58)
R^2: 0.34326197788973173
0.4
(2138004, 58)
R^2: 0.3454423019905113
0.45
(2138004, 58)
R^2: 0.3454288555255267
0.5
(2138004, 58)
R^2: 0.3526048478829526
0.55
(2138004, 58)
R^2: 0.33529664471926013
0.6000000000000001
(2138004, 58)
R^2: 0.3448016234472867
0.65
(2138004, 58)
R^2: 0.3437462276954097
0.7000000000000001
(2138004, 58)
R^2: 0.33462647323491784
0.75
(2138004, 58)
R^2: 0.338347793334673

In [3]:
rf = RandomForestRegressor(n_estimators=20, max_depth=5, random_state=0, verbose=2, n_jobs=-1)
rf.fit(X_train, y_t_train)
rf

pred = rf.predict(X_test)           # -> (n_test, 3)
print(pred.shape)
try:
    print("R^2:", rf.score(X_test, y_t_test))
except:
    print('woops')

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 25 concurrent workers.


building tree 1 of 20
building tree 2 of 20
building tree 3 of 20
building tree 4 of 20
building tree 5 of 20
building tree 6 of 20
building tree 7 of 20
building tree 8 of 20
building tree 9 of 20
building tree 10 of 20
building tree 11 of 20
building tree 12 of 20
building tree 13 of 20
building tree 14 of 20
building tree 15 of 20
building tree 16 of 20
building tree 17 of 20
building tree 18 of 20
building tree 19 of 20
building tree 20 of 20


[Parallel(n_jobs=-1)]: Done   4 out of  20 | elapsed:  8.5min remaining: 34.2min
[Parallel(n_jobs=-1)]: Done  15 out of  20 | elapsed:  8.6min remaining:  2.9min
[Parallel(n_jobs=-1)]: Done  20 out of  20 | elapsed:  8.6min finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   3 out of  20 | elapsed:    0.9s remaining:    5.3s
[Parallel(n_jobs=20)]: Done  14 out of  20 | elapsed:    1.5s remaining:    0.7s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    1.9s finished


(2138004, 58)


[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   3 out of  20 | elapsed:    1.0s remaining:    5.4s
[Parallel(n_jobs=20)]: Done  14 out of  20 | elapsed:    1.5s remaining:    0.7s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    1.9s finished


R^2: 0.18892544897563568


In [4]:
rf = RandomForestRegressor(n_estimators=100, random_state=0, max_depth=20, verbose=2, n_jobs=-1)
rf.fit(X_train, y_t_train)
rf

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 25 concurrent workers.


building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
b

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 126.4min finished


,n_estimators,100
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [5]:
pred = rf.predict(X_test)           # -> (n_test, 3)
print(pred.shape)
try:
    print("R^2:", rf.score(X_test, y_t_test))
except:
    print('woops')

[Parallel(n_jobs=25)]: Using backend ThreadingBackend with 25 concurrent workers.
[Parallel(n_jobs=25)]: Done 100 out of 100 | elapsed:    6.8s finished


(2138004, 58)


[Parallel(n_jobs=25)]: Using backend ThreadingBackend with 25 concurrent workers.
[Parallel(n_jobs=25)]: Done 100 out of 100 | elapsed:    6.9s finished


R^2: 0.40361490185813326


In [6]:
np.save(os.path.join(test_folder, "y_t_pred_rf1.npy"), pred)

In [7]:
rf = RandomForestRegressor(n_estimators=100, random_state=0, max_depth=20, verbose=2, n_jobs=-1)
rf.fit(X_train, y_q_train)
rf

pred = rf.predict(X_test)           # -> (n_test, 3)
print(pred.shape)
try:
    print("R^2:", rf.score(X_test, y_q_test))
except:
    print('woops')


np.save(os.path.join(test_folder, "y_q_pred_rf1.npy"), pred)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 25 concurrent workers.


building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
b

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 57.1min finished
[Parallel(n_jobs=25)]: Using backend ThreadingBackend with 25 concurrent workers.
[Parallel(n_jobs=25)]: Done 100 out of 100 | elapsed:    6.7s finished


(2138004, 58)


[Parallel(n_jobs=25)]: Using backend ThreadingBackend with 25 concurrent workers.
[Parallel(n_jobs=25)]: Done 100 out of 100 | elapsed:    6.8s finished


R^2: 0.3246682310481606


In [ ]:
param_dist = {
    "n_estimators":      [100, 200, 400, 800],
    "max_depth":         [None, 10, 20, 40],
    "min_samples_leaf":  [1, 2, 5, 10],
    "max_features":      ["sqrt", 0.3, 0.5, 1.0],
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=0, n_jobs=-1),
    param_dist, n_iter=20, cv=3,
    scoring="r2", random_state=0, n_jobs=-1,
)
search.fit(X_train, y_t_train)
print(search.best_params_, search.best_score_)

best = search.best_estimator_
joblib.dump(best, os.path.join(data_folder, "models/rf_model_t.joblib"))

In [ ]:
y_t_pred = best.predict(X_test)
np.save(os.path.join(test_folder, "y_t_pred_rf2.npy"), y_t_pred)

In [ ]:
param_dist = {
    "n_estimators":      [100, 200, 400, 800],
    "max_depth":         [None, 10, 20, 40],
    "min_samples_leaf":  [1, 2, 5, 10],
    "max_features":      ["sqrt", 0.3, 0.5, 1.0],
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=0, n_jobs=-1),
    param_dist, n_iter=20, cv=3,
    scoring="r2", random_state=0, n_jobs=-1,
)
search.fit(X_train, y_q_train)
print(search.best_params_, search.best_score_)

best = search.best_estimator_
joblib.dump(best, os.path.join(data_folder, "models/rf_model_q.joblib"))

In [ ]:
y_q_pred = best.predict(X_test)
np.save(os.path.join(test_folder, "y_q_pred_rf2.npy"), y_q_pred)

In [ ]:
import time
sub = 50_000                      # small slice
t = time.time()
RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0, max_depth=15, verbose=2).fit(
    X_train[:sub], y_t_train[:sub])
dt = time.time() - t
# trees scale linearly; rows scale ~n·log(n)
import math
factor = (1_000_000 * math.log(1_000_000)) / (sub * math.log(sub))
print(f"slice took {dt:.0f}s → full set roughly {dt*factor:.0f}s")